# 混合记忆：向量+图+KV

Mem0将记忆作为并行的三个数据库————向量用于语义相似度、KV用于快速事实查询、图用于实体关系推理，再由打分层在检索时融合三者。

## 问题描述

对三类查询的某一类来说，单一存储是错的：
- 语义相似度 ———— 向量赢，KV和图不行。
- 事实查询 ———— KV赢，向量太浪费了，图大材小用。
- 关系推理 ———— 图赢，向量和KV答不了

生产级agents在一个会话里这三类都会发出来。单一存储记忆对其中两类总是错的。Mem0的贡献是将这三种都接在一个`add/search`接口后面，然后使用一个打分函数进行融合。

## 基本概念

### 并行的三个数据库

Mem0 的`add(text, user_id, metadata)`实现：
1. 从文本中抽取候选事实。（步骤由LLM驱动）
2. 将每个事实写到向量储存（嵌入），用做后续的语义搜索
3. 将每个事实组合`(user_id, fact_type, entity)`键写到KV储存，供常数时间查询。
4. 将每个事实作为带类型的边写进图储存，供关系查询。

`search（query, user_id）`实现：
1. 向量存储返回嵌入余弦相似度最高的k个候选。
2. KV存储返回组合键查到的内容。
3. 图存储返回从query能够摸到的子图。
4. 一个打分层融合三者。

### 融合打分
- 相关性。  向量余弦、KV精确匹配、图边权重。
- 重要性。  写入时标记或者学习获得（一些比较重要的事实：名称、IDs、政策）。
- 新鲜度。  自上次写入或读取以来指数衰减。

权重需要跟随产品进行调整。对于chat agents，需要更高的新鲜度；对于合规agent，需要更高的重要性；对于检索agent，需要更高的相关性。

### Mem0g 和时序推理

Mem0g 加了一个冲突检测器。当一个新的事实与已存在的边冲突时，已存在的边会被标记为失效而非删除。时序查询（“三月份的时候用户在哪？”）只会遍历到当时有效的子图。

### 范围分类

Mem0 将记忆分类：
- 用户记忆。  会话间持久，通过`user_id`标识
- 会话记忆。  一个进程内持久。
- Agent记忆。 存在于每个agent实例。

每次写都需要选一个范围，检索的时候跨范围并加权。

### 什么时候失效

- 嵌入漂移。  前一百次查询看起来对的结果，会随语料增长而退化。加上对使用最频繁的top-n条记录的周期性重嵌入。
- KV schema 蔓延。 `(user_id, type, entity)`看起来简单但是每个团队都会加自己的的`type`。定时对type集合审计。
- 图爆炸。 含噪的抽取器一条信息会添加50条边，给每次add调用写入图的元素做上限封顶，丢弃置信度最低的边。

# 开始编码

对应本章核心：**向量 + KV + 图三库并行**、**统一 `add/search` + 融合打分**、**冲突失效（Mem0g）与范围（user/session/agent）**。  
先用玩具跑通三路写入与融合检索；再用 **PyTorch** 学融合权重；最后用 **LangChain + DeepSeek** 挂上真实 `memory_add` / `memory_search`。


## 1. 教学玩具：Mem0 式混合记忆

- **Vector**：词袋余弦（语义近似）。
- **KV**：`(scope_id, fact_type, entity)` 常数时间查事实。
- **Graph**：带类型边；冲突边标记 `invalid` 而非删除（时序可查）。
- **Fusion**：相关性 × 重要性 × 新鲜度加权求和。


In [ ]:
from __future__ import annotations

import math
import re
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Literal

Scope = Literal["user", "session", "agent"]


def _tokenize(text: str) -> set[str]:
    """简易分词（中英混合）。"""
    return set(re.findall(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]+", text.lower()))


def bag_cosine(a: str, b: str) -> float:
    """
    Args:
        a: 文本 A。
        b: 文本 B。

    Returns:
        score: 词袋余弦相似度 ∈ [0, 1]。
    """
    ta, tb = _tokenize(a), _tokenize(b)
    if not ta or not tb:
        return 0.0
    inter = len(ta & tb)
    return inter / math.sqrt(len(ta) * len(tb))


@dataclass
class MemoryFact:
    """一条抽取后的事实（三库共享同一 id）。"""

    fact_id: str
    text: str
    scope: Scope
    scope_id: str
    fact_type: str
    entity: str
    importance: float
    created_at: float
    last_access_at: float
    valid: bool = True
    invalid_at: float | None = None


@dataclass
class VectorHit:
    """向量检索命中。"""

    fact: MemoryFact
    score: float


@dataclass
class VectorStore:
    """语义相似度存储（玩具：词袋余弦）。"""

    facts: dict[str, MemoryFact] = field(default_factory=dict)

    def upsert(self, fact: MemoryFact) -> None:
        """写入 / 覆盖。"""
        self.facts[fact.fact_id] = fact

    def search(self, query: str, *, scope_id: str | None, top_k: int = 5) -> list[VectorHit]:
        """
        Args:
            query: 查询。
            scope_id: 可选范围过滤。
            top_k: 返回条数。

        Returns:
            hits: 按余弦降序。
        """
        scored: list[VectorHit] = []
        for fact in self.facts.values():
            if not fact.valid:
                continue
            if scope_id is not None and fact.scope_id != scope_id:
                continue
            scored.append(VectorHit(fact, bag_cosine(query, fact.text)))
        scored.sort(key=lambda h: h.score, reverse=True)
        return [h for h in scored if h.score > 0][:top_k]


@dataclass
class KVStore:
    """快速事实查询：组合键 → fact_id。"""

    table: dict[tuple[str, str, str], str] = field(default_factory=dict)
    facts: dict[str, MemoryFact] = field(default_factory=dict)

    def key(self, scope_id: str, fact_type: str, entity: str) -> tuple[str, str, str]:
        """
        Returns:
            key: ``(scope_id, fact_type, entity)``。
        """
        return (scope_id, fact_type.lower(), entity.lower())

    def upsert(self, fact: MemoryFact) -> str | None:
        """
        Args:
            fact: 新事实。

        Returns:
            old_fact_id: 若键冲突则返回旧 id，否则 ``None``。
        """
        k = self.key(fact.scope_id, fact.fact_type, fact.entity)
        old = self.table.get(k)
        self.table[k] = fact.fact_id
        self.facts[fact.fact_id] = fact
        return old

    def get(self, scope_id: str, fact_type: str, entity: str) -> MemoryFact | None:
        """
        Args:
            scope_id: 范围 id。
            fact_type: 事实类型。
            entity: 实体。

        Returns:
            fact: 命中且仍 valid 的事实。
        """
        fid = self.table.get(self.key(scope_id, fact_type, entity))
        if fid is None:
            return None
        fact = self.facts.get(fid)
        if fact is None or not fact.valid:
            return None
        return fact


@dataclass
class GraphEdge:
    """带类型边；冲突时 ``valid=False``（Mem0g）。"""

    edge_id: str
    src: str
    rel: str
    dst: str
    fact_id: str
    weight: float
    valid: bool = True
    valid_from: float = 0.0
    valid_to: float | None = None


@dataclass
class GraphStore:
    """实体关系图；冲突边失效而非删除。"""

    edges: list[GraphEdge] = field(default_factory=list)
    node_facts: dict[str, list[str]] = field(default_factory=dict)

    def add_edge(self, edge: GraphEdge, *, conflict_rel_same_src: bool = True) -> list[str]:
        """
        Args:
            edge: 新边。
            conflict_rel_same_src: 同 src+rel 不同 dst 视为冲突。

        Returns:
            invalidated: 被标记失效的边 id 列表。
        """
        invalidated: list[str] = []
        if conflict_rel_same_src:
            for e in self.edges:
                if (
                    e.valid
                    and e.src == edge.src
                    and e.rel == edge.rel
                    and e.dst != edge.dst
                ):
                    e.valid = False
                    e.valid_to = edge.valid_from
                    invalidated.append(e.edge_id)
        self.edges.append(edge)
        self.node_facts.setdefault(edge.src, []).append(edge.fact_id)
        self.node_facts.setdefault(edge.dst, []).append(edge.fact_id)
        return invalidated

    def neighborhood(
        self,
        query: str,
        *,
        at_time: float | None = None,
        top_k: int = 5,
    ) -> list[tuple[float, GraphEdge]]:
        """
        Args:
            query: 用实体词触摸子图。
            at_time: 时序点；``None`` 表示只要当前 valid。
            top_k: 条数。

        Returns:
            hits: ``(score, edge)``。
        """
        toks = _tokenize(query)
        scored: list[tuple[float, GraphEdge]] = []
        for e in self.edges:
            if at_time is None:
                if not e.valid:
                    continue
            else:
                if e.valid_from > at_time:
                    continue
                if e.valid_to is not None and e.valid_to <= at_time:
                    continue
            nodes = _tokenize(e.src) | _tokenize(e.rel) | _tokenize(e.dst)
            # 子串匹配：query "lives" 可摸到边 rel "lives_in"
            hit = 0
            for t in toks:
                if any(t == n or t in n or n in t for n in nodes):
                    hit += 1
            overlap = hit / max(len(toks), 1)
            if overlap > 0:
                scored.append((overlap * e.weight, e))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]


@dataclass
class FusionWeights:
    """融合打分权重（可随产品调）。"""

    relevance: float = 0.5
    importance: float = 0.3
    freshness: float = 0.2
    vector: float = 0.5
    kv: float = 0.3
    graph: float = 0.2


@dataclass
class ScoredMemory:
    """融合后的检索结果。"""

    fact: MemoryFact
    score: float
    channels: dict[str, float]


@dataclass
class HybridMemory:
    """Mem0 风格：三库并行 + 统一 add/search。"""

    vector: VectorStore = field(default_factory=VectorStore)
    kv: KVStore = field(default_factory=KVStore)
    graph: GraphStore = field(default_factory=GraphStore)
    facts: dict[str, MemoryFact] = field(default_factory=dict)
    weights: FusionWeights = field(default_factory=FusionWeights)
    freshness_half_life_s: float = 3600.0
    max_edges_per_add: int = 3

    def _freshness(self, fact: MemoryFact, now: float | None = None) -> float:
        """
        Args:
            fact: 事实。
            now: 当前时间戳。

        Returns:
            fresh: 指数衰减新鲜度 ∈ (0, 1]。
        """
        now = time.time() if now is None else now
        age = max(now - fact.last_access_at, 0.0)
        return 0.5 ** (age / max(self.freshness_half_life_s, 1.0))

    def _extract_facts(self, text: str) -> list[dict[str, Any]]:
        """
        玩具抽取器（生产由 LLM 驱动）。

        Args:
            text: 原始文本。

        Returns:
            candidates: 候选事实字典列表。
        """
        cands: list[dict[str, Any]] = []
        # name=Ada / 叫 Ada
        for m in re.finditer(r"(?:name\s*=\s*|叫\s*|我是\s*)([A-Za-z\u4e00-\u9fff]+)", text):
            cands.append(
                {
                    "text": f"name is {m.group(1)}",
                    "fact_type": "name",
                    "entity": "user",
                    "importance": 0.95,
                    "rel": ("user", "named", m.group(1)),
                }
            )
        for m in re.finditer(r"prefer[s]?\s+(\w+)", text, flags=re.I):
            cands.append(
                {
                    "text": f"prefers {m.group(1)}",
                    "fact_type": "preference",
                    "entity": "user",
                    "importance": 0.8,
                    "rel": ("user", "prefers", m.group(1)),
                }
            )
        for m in re.finditer(r"(?:lives?\s+in|居住在|住在)\s*([A-Za-z\u4e00-\u9fff]+)", text, flags=re.I):
            cands.append(
                {
                    "text": f"lives in {m.group(1)}",
                    "fact_type": "location",
                    "entity": "user",
                    "importance": 0.85,
                    "rel": ("user", "lives_in", m.group(1)),
                }
            )
        for m in re.finditer(r"(?:works? at|就职于)\s*([A-Za-z0-9_\u4e00-\u9fff]+)", text, flags=re.I):
            cands.append(
                {
                    "text": f"works at {m.group(1)}",
                    "fact_type": "employer",
                    "entity": "user",
                    "importance": 0.8,
                    "rel": ("user", "works_at", m.group(1)),
                }
            )
        for m in re.finditer(r"policy:\s*([^\n。]+)", text, flags=re.I):
            cands.append(
                {
                    "text": f"policy: {m.group(1).strip()}",
                    "fact_type": "policy",
                    "entity": "org",
                    "importance": 0.99,
                    "rel": ("org", "policy", m.group(1).strip()[:24]),
                }
            )
        # 兜底：整句作为语义事实
        if not cands:
            cands.append(
                {
                    "text": text.strip(),
                    "fact_type": "note",
                    "entity": "misc",
                    "importance": 0.4,
                    "rel": ("misc", "notes", text.strip()[:24] or "empty"),
                }
            )
        return cands[: self.max_edges_per_add]

    def add(
        self,
        text: str,
        *,
        scope: Scope = "user",
        scope_id: str = "default",
        metadata: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        """
        并行写入三库。

        Args:
            text: 原始文本。
            scope: ``user`` / ``session`` / ``agent``。
            scope_id: 范围标识（如 user_id）。
            metadata: 可选元数据（可覆盖 importance）。

        Returns:
            report: 写入与冲突失效摘要。
        """
        metadata = metadata or {}
        now = time.time()
        written: list[str] = []
        invalidated: list[str] = []
        for cand in self._extract_facts(text):
            importance = float(metadata.get("importance", cand["importance"]))
            fact = MemoryFact(
                fact_id=uuid.uuid4().hex[:10],
                text=str(cand["text"]),
                scope=scope,
                scope_id=scope_id,
                fact_type=str(cand["fact_type"]),
                entity=str(cand["entity"]),
                importance=importance,
                created_at=now,
                last_access_at=now,
            )
            # KV 冲突 → 旧事实失效（Mem0g）
            old_id = self.kv.upsert(fact)
            if old_id and old_id in self.facts:
                old = self.facts[old_id]
                old.valid = False
                old.invalid_at = now
                invalidated.append(old_id)
            self.vector.upsert(fact)
            self.facts[fact.fact_id] = fact
            src, rel, dst = cand["rel"]
            edge = GraphEdge(
                edge_id=uuid.uuid4().hex[:8],
                src=str(src),
                rel=str(rel),
                dst=str(dst),
                fact_id=fact.fact_id,
                weight=importance,
                valid_from=now,
            )
            invalidated.extend(self.graph.add_edge(edge))
            written.append(fact.fact_id)
        return {
            "written": written,
            "invalidated": invalidated,
            "scope": scope,
            "scope_id": scope_id,
            "n_edges_capped": self.max_edges_per_add,
        }

    def search(
        self,
        query: str,
        *,
        scope_id: str | None = None,
        top_k: int = 5,
        at_time: float | None = None,
        kv_hint: tuple[str, str] | None = None,
    ) -> list[ScoredMemory]:
        """
        三路召回 + 融合打分。

        Args:
            query: 查询。
            scope_id: 可选范围。
            top_k: 返回条数。
            at_time: 图时序点。
            kv_hint: 可选 ``(fact_type, entity)`` 精确查。

        Returns:
            results: 融合后排序列表。
        """
        w = self.weights
        channel: dict[str, dict[str, float]] = {}

        def _acc(fid: str, ch: str, score: float) -> None:
            channel.setdefault(fid, {})
            channel[fid][ch] = max(channel[fid].get(ch, 0.0), score)

        for hit in self.vector.search(query, scope_id=scope_id, top_k=top_k * 2):
            _acc(hit.fact.fact_id, "vector", hit.score)

        if kv_hint is not None and scope_id is not None:
            fact = self.kv.get(scope_id, kv_hint[0], kv_hint[1])
            if fact is not None:
                _acc(fact.fact_id, "kv", 1.0)
        else:
            # 弱 KV：从 query 猜 type/entity
            for fact_type in ("name", "preference", "location", "employer", "policy"):
                if fact_type[:4] in query.lower() or fact_type in query.lower():
                    if scope_id is not None:
                        fact = self.kv.get(scope_id, fact_type, "user")
                        if fact is None:
                            fact = self.kv.get(scope_id, fact_type, "org")
                        if fact is not None:
                            _acc(fact.fact_id, "kv", 1.0)

        for score, edge in self.graph.neighborhood(query, at_time=at_time, top_k=top_k * 2):
            _acc(edge.fact_id, "graph", float(score))

        # 时序模式：把当时仍有效的事实直接纳入（即使现已 invalid）
        if at_time is not None:
            for fact in self.facts.values():
                if fact.created_at > at_time:
                    continue
                if fact.invalid_at is not None and fact.invalid_at <= at_time:
                    continue
                # 弱语义门槛，避免全库灌入
                sim = bag_cosine(query, fact.text)
                if sim > 0:
                    _acc(fact.fact_id, "vector", sim)

        now = time.time()
        fused: list[ScoredMemory] = []
        for fid, ch in channel.items():
            fact = self.facts.get(fid)
            if fact is None:
                continue
            if at_time is None and not fact.valid:
                continue
            if at_time is not None:
                if fact.created_at > at_time:
                    continue
                if fact.invalid_at is not None and fact.invalid_at <= at_time:
                    continue
            rel = (
                w.vector * ch.get("vector", 0.0)
                + w.kv * ch.get("kv", 0.0)
                + w.graph * ch.get("graph", 0.0)
            )
            fresh = self._freshness(fact, now=now if at_time is None else at_time)
            score = w.relevance * rel + w.importance * fact.importance + w.freshness * fresh
            fact.last_access_at = now
            fused.append(ScoredMemory(fact=fact, score=score, channels=ch))
        fused.sort(key=lambda x: x.score, reverse=True)
        return fused[:top_k]


# 冒烟
_m = HybridMemory()
_r = _m.add("我是 Ada，prefers concise，lives in Shanghai", scope_id="u1")
assert _r["written"]
_hits = _m.search("where does user live", scope_id="u1")
assert any("Shanghai" in h.fact.text for h in _hits)
print("HybridMemory toy ready | add/search OK | scopes=user|session|agent")


## 2. 玩具示例：三路查询 + 冲突失效 + 时序子图 + 范围


In [ ]:
def demo_hybrid_memory() -> None:
    """语义 / 事实 / 关系三类查询；location 冲突后旧边失效；时序可回看。"""
    mem = HybridMemory(freshness_half_life_s=10_000)
    # 调高相关性，方便看通道差异
    mem.weights = FusionWeights(relevance=0.6, importance=0.25, freshness=0.15)

    t0 = time.time()
    mem.add("我是 Ada，prefers concise，lives in Beijing", scope="user", scope_id="u1")
    time.sleep(0.01)
    t_mid = time.time()
    # 冲突：搬家 → 旧 lives_in 边失效，KV 覆盖
    mem.add("Ada lives in Shanghai", scope="user", scope_id="u1")
    mem.add("works at DeepSeek", scope="user", scope_id="u1")
    mem.add("policy: never share API keys", scope="agent", scope_id="agent-main")
    mem.add("session note: debugging hybrid fusion", scope="session", scope_id="s9")

    print("=== semantic (vector-heavy) ===")
    for h in mem.search("what writing style does Ada like", scope_id="u1"):
        print(f"{h.score:.3f} {h.channels} | {h.fact.text}")

    print("\n=== factual KV hint ===")
    for h in mem.search("employer", scope_id="u1", kv_hint=("employer", "user")):
        print(f"{h.score:.3f} {h.channels} | {h.fact.text}")

    print("\n=== relational graph ===")
    for h in mem.search("where user lives_in", scope_id="u1"):
        print(f"{h.score:.3f} {h.channels} | {h.fact.text} valid={h.fact.valid}")

    # 当前应是 Shanghai
    cur = mem.search("lives", scope_id="u1")
    assert any("Shanghai" in h.fact.text and h.fact.valid for h in cur)

    print("\n=== temporal subgraph @ t_mid (before move) ===")
    past = mem.search("lives", scope_id="u1", at_time=t_mid)
    for h in past:
        print(f"{h.score:.3f} | {h.fact.text} valid_now={h.fact.valid}")
    assert any("Beijing" in h.fact.text for h in past)

    # 失效边仍在图里
    invalid_edges = [e for e in mem.graph.edges if not e.valid and e.rel == "lives_in"]
    assert invalid_edges, "conflict should invalidate old edge"
    print("\ninvalidated lives_in edges:", len(invalid_edges))

    # 范围：agent policy 不应被 user scope_id 过滤掉时显式查 agent
    agent_hits = mem.search("API keys policy", scope_id="agent-main")
    assert any("API" in h.fact.text or "policy" in h.fact.text.lower() for h in agent_hits)
    print("\n=== agent scope ===")
    for h in agent_hits[:3]:
        print(h.fact.scope, h.fact.text)

    print("TOY DEMO OK")


demo_hybrid_memory()


## 3. PyTorch：可学习融合权重

把三通道分（vector / kv / graph）与 importance、freshness 喂入小 MLP，学出最终排序分——对应笔记里「权重跟随产品调整」。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FusionNet(nn.Module):
    """五维通道特征 → 融合分。"""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(5,)`` 或 ``(B, 5)`` —— vector, kv, graph, importance, freshness。

        Returns:
            score: 标量或 ``(B,)``。
        """
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        y = self.net(x).squeeze(-1)
        return y.squeeze(0) if single else y


def pack_features(
    vector: float,
    kv: float,
    graph: float,
    importance: float,
    freshness: float,
) -> torch.Tensor:
    """
    Returns:
        x: ``(5,)`` float tensor。
    """
    return torch.tensor([vector, kv, graph, importance, freshness], dtype=torch.float32)


def train_fusion_net(
    pairs: list[tuple[torch.Tensor, torch.Tensor]],
    *,
    steps: int = 400,
    lr: float = 0.05,
) -> FusionNet:
    """
    用 pairwise hinge：正例分应高于负例。

    Args:
        pairs: ``(pos_feat, neg_feat)``。
        steps: 步数。
        lr: 学习率。

    Returns:
        model: 训练后融合网。
    """
    model = FusionNet()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for pos, neg in pairs:
            sp, sn = model(pos), model(neg)
            loss = loss + F.relu(0.5 + sn - sp)
        loss = loss / max(len(pairs), 1)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


@torch.no_grad()
def rank_with_net(model: FusionNet, rows: list[tuple[str, torch.Tensor]]) -> list[tuple[float, str]]:
    """
    Args:
        model: 融合网。
        rows: ``(label, features)``。

    Returns:
        ranked: ``(score, label)`` 降序。
    """
    scored = [(float(model(x).item()), label) for label, x in rows]
    scored.sort(key=lambda t: t[0], reverse=True)
    return scored


def demo_pytorch_fusion() -> None:
    """学到：精确 KV / 高重要政策 排在噪声语义之上。"""
    torch.manual_seed(0)
    pairs = [
        (
            pack_features(0.3, 1.0, 0.2, 0.9, 0.8),  # 强 KV 事实
            pack_features(0.7, 0.0, 0.0, 0.3, 0.5),  # 纯语义噪声
        ),
        (
            pack_features(0.2, 0.0, 0.9, 0.85, 0.7),  # 强图关系
            pack_features(0.6, 0.0, 0.1, 0.3, 0.4),
        ),
        (
            pack_features(0.4, 0.5, 0.3, 0.99, 0.6),  # 高重要性政策
            pack_features(0.8, 0.0, 0.0, 0.2, 0.9),
        ),
    ]
    model = train_fusion_net(pairs)
    ranked = rank_with_net(
        model,
        [
            ("kv_fact", pack_features(0.25, 1.0, 0.1, 0.9, 0.75)),
            ("semantic_noise", pack_features(0.75, 0.0, 0.0, 0.25, 0.5)),
            ("policy", pack_features(0.35, 0.4, 0.2, 0.99, 0.55)),
        ],
    )
    print("=== learned fusion ranking ===")
    for s, label in ranked:
        print(f"{s:.3f}  {label}")
    assert ranked[0][1] in {"kv_fact", "policy"}
    assert ranked[-1][1] == "semantic_noise"
    print("PYTORCH DEMO OK")


demo_pytorch_fusion()


## 4. 生产级：LangChain Hybrid Memory 工具 + DeepSeek

``memory_add`` / ``memory_search`` 封成工具；agent 按问题类型触发写入或三路融合检索。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

PROD_MEM = HybridMemory(freshness_half_life_s=86_400)
PROD_USER = "user-ada"


class MemoryAddArgs(BaseModel):
    """memory_add 参数。"""

    text: str = Field(description="Raw text to extract and store as hybrid memory")
    scope: Literal["user", "session", "agent"] = Field(
        default="user", description="Memory scope"
    )
    scope_id: str = Field(default=PROD_USER, description="Scope id, e.g. user_id")
    importance: float | None = Field(
        default=None, ge=0.0, le=1.0, description="Optional importance override"
    )


class MemorySearchArgs(BaseModel):
    """memory_search 参数。"""

    query: str = Field(description="Search query")
    scope_id: str = Field(default=PROD_USER, description="Scope id to filter")
    top_k: int = Field(default=5, ge=1, le=10)
    fact_type: str | None = Field(
        default=None, description="Optional KV fact_type hint, e.g. location"
    )
    entity: str | None = Field(default=None, description="Optional KV entity hint")


def memory_add_impl(
    text: str,
    scope: str = "user",
    scope_id: str = PROD_USER,
    importance: float | None = None,
) -> str:
    """
    Args:
        text: 原始文本。
        scope: 范围。
        scope_id: 范围 id。
        importance: 可选重要性。

    Returns:
        observation: JSON 写入报告。
    """
    meta = {"importance": importance} if importance is not None else {}
    report = PROD_MEM.add(text, scope=scope, scope_id=scope_id, metadata=meta)  # type: ignore[arg-type]
    return json.dumps(report, ensure_ascii=False)


def memory_search_impl(
    query: str,
    scope_id: str = PROD_USER,
    top_k: int = 5,
    fact_type: str | None = None,
    entity: str | None = None,
) -> str:
    """
    Args:
        query: 查询。
        scope_id: 范围。
        top_k: 条数。
        fact_type: KV 类型提示。
        entity: KV 实体提示。

    Returns:
        observation: JSON 命中列表。
    """
    hint = (fact_type, entity) if fact_type and entity else None
    hits = PROD_MEM.search(query, scope_id=scope_id, top_k=top_k, kv_hint=hint)
    payload = [
        {
            "score": round(h.score, 4),
            "text": h.fact.text,
            "channels": h.channels,
            "scope": h.fact.scope,
            "type": h.fact.fact_type,
            "importance": h.fact.importance,
            "valid": h.fact.valid,
        }
        for h in hits
    ]
    return json.dumps({"hits": payload}, ensure_ascii=False)


def build_memory_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: add/search 两个工具。
    """

    def _add(**kwargs: Any) -> str:
        args = MemoryAddArgs(**kwargs)
        return memory_add_impl(
            args.text, args.scope, args.scope_id, args.importance
        )

    def _search(**kwargs: Any) -> str:
        args = MemorySearchArgs(**kwargs)
        return memory_search_impl(
            args.query, args.scope_id, args.top_k, args.fact_type, args.entity
        )

    return [
        StructuredTool.from_function(
            name="memory_add",
            description=(
                "Store text into hybrid memory (vector+KV+graph). "
                "Use when user shares lasting facts."
            ),
            func=_add,
            args_schema=MemoryAddArgs,
        ),
        StructuredTool.from_function(
            name="memory_search",
            description=(
                "Search hybrid memory with fused vector/KV/graph scores. "
                "Use for semantic, factual, or relational questions about stored memory."
            ),
            func=_search,
            args_schema=MemorySearchArgs,
        ),
    ]


MEMORY_TOOLS = build_memory_tools()


def reset_prod_memory() -> None:
    """重置生产记忆。"""
    global PROD_MEM, MEMORY_TOOLS
    PROD_MEM = HybridMemory(freshness_half_life_s=86_400)
    MEMORY_TOOLS = build_memory_tools()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def build_hybrid_agent() -> Any:
    """
    Returns:
        agent: 带混合记忆工具的 agent。
    """
    system = (
        "You are an agent with Mem0-style hybrid memory (vector + KV + graph).\n"
        "When the user states lasting facts, call memory_add.\n"
        "When answering about past facts, call memory_search first "
        "(use fact_type/entity hints for exact lookups like location/name).\n"
        "Reply in Chinese, concise."
    )
    return create_agent(get_llm(), MEMORY_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """
    Args:
        messages: 轨迹。

    Returns:
        text: 可读摘要。
    """
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args')})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"OBS[{m.name}]: {m.content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    """
    Returns:
        n: 工具调用次数。
    """
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def run_hybrid_turn(user_text: str) -> dict[str, Any]:
    """
    Args:
        user_text: 用户输入。

    Returns:
        result: agent 返回值。
    """
    return build_hybrid_agent().invoke({"messages": [HumanMessage(content=user_text)]})


print(f"LangChain HybridMemory ready | {MODEL}")


## 5. 生产示例：写入事实 → 搬家冲突 → 融合检索


In [ ]:
def demo_deepseek_hybrid_memory() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod_memory()

    r1 = run_hybrid_turn(
        "请用 memory_add 记住：我叫 Ada，prefers concise，lives in Beijing，works at DeepSeek。"
    )
    print("=== add turn ===")
    print(format_agent_messages(r1["messages"]))
    assert count_tool_calls(r1["messages"]) >= 1
    assert len(PROD_MEM.facts) >= 1

    r2 = run_hybrid_turn("我搬家了：lives in Shanghai。请 memory_add 更新。")
    print("\n=== conflict update ===")
    print(format_agent_messages(r2["messages"]))

    r3 = run_hybrid_turn(
        "用 memory_search 回答：我现在住在哪？如需精确查找可用 fact_type=location, entity=user。"
    )
    print("\n=== search turn ===")
    print(format_agent_messages(r3["messages"]))
    blob = format_agent_messages(r3["messages"]).lower()
    assert "shanghai" in blob or "上海" in blob
    # 图上应有失效的旧边或无效旧事实
    assert any(not e.valid for e in PROD_MEM.graph.edges) or any(
        not f.valid for f in PROD_MEM.facts.values()
    )
    print("\nPRODUCTION DEMO OK")


demo_deepseek_hybrid_memory()
